In [ ]:
import sys
sys.path.append("../")

In [ ]:
import geopandas as gpd
import folium
from shapely import Polygon
from shapely.geometry import mapping

import src.constants as CONST
import src.paths as PATHS

In [ ]:
sam_dir = PATHS.DATA_DIR / "sam"

In [ ]:
brakel_gdf = gpd.read_file(sam_dir / "Brakel.geojsonl")

In [ ]:
brakel_gdf.info(memory_usage="deep")

In [ ]:
brakel_gdf.head(10)

In [ ]:
brakel_gdf.iloc[1]["geometry"]

In [ ]:
brakel_gdf.loc[445:455]

In [ ]:
draft_data = PATHS.DATA_DIR / "phase1_2025-08-14_v1.gpkg"

gpkg_dict = {}
for _, layer_data in gpd.list_layers(draft_data).iterrows():
    gpkg_dict[layer_data["name"]] = gpd.read_file(draft_data, layer=layer_data["name"])

In [ ]:
gpkg_dict.keys()

In [ ]:
shape1 = list(brakel_gdf.iloc[0]['geometry'].exterior.coords)

In [ ]:
shape2 = list(brakel_gdf.iloc[1]['geometry'].exterior.coords)

In [ ]:
example_poly = Polygon(list(set(shape1).intersection(set(shape2)))).convex_hull

In [ ]:
# Center the map on the polygon
center = example_poly.centroid.coords[0]
m = folium.Map(location=[center[1], center[0]], zoom_start=12, tiles="cartodb_positron", control_scale=True)

# Add the polygon
folium.GeoJson(mapping(example_poly)).add_to(m)

folium.GeoJson(mapping(brakel_gdf.iloc[450]['geometry']),
               style_function=lambda x: {
                   "color": "red",         # border color
                   "fillColor": "red",    # fill color
                   "weight": 2,            # border thickness
                   "fillOpacity": 0.3      # fill opacity
    }
).add_to(m)

folium.GeoJson(mapping(brakel_gdf.iloc[451]['geometry']),
               style_function=lambda x: {
                   "color": "green",         # border color
                   "fillColor": "green",    # fill color
                   "weight": 2,            # border thickness
                   "fillOpacity": 0.3      # fill opacity
    }
).add_to(m)

for layer in gpkg_dict:
    if layer == "vlakken_scope":
        fg = folium.FeatureGroup(name=layer, show=True).add_to(m)
        folium.GeoJson(gpkg_dict[layer]["geometry"].to_crs(epsg=CONST.EPSG_WGS84)).add_to(fg)

fg = folium.FeatureGroup(name="Selected floodplains", show=False).add_to(m)

folium.LayerControl().add_to(m)

m